#### **Q4. Viterbi Algorithm Implementation for the Nature Primer**

# Viterbi Algorithm for a Simple HMM

This Python script implements the **Viterbi algorithm** to find the most probable sequence of hidden states in a simplified **Hidden Markov Model (HMM)** for gene annotation. It also calculates the log-probability of a manually specified state path.

---

## States

We consider three hidden states:

- `E`: Exon (uniform base emission)
- `5`: 5' splice site (strongly favors `G`)
- `I`: Intron (favors `A` and `T`)

---

## Transition Probabilities

```python
transition_probs = {
    'Start': {'E': 1.0},
    'E': {'E': 0.9, '5': 0.1},
    '5': {'I': 1.0},
    'I': {'I': 0.9, 'End': 0.1}
}


In [7]:
import numpy as np
import math

# Helper function to calculate
def log(x):
    return -math.inf if x == 0 else math.log(x)

def get_log_prob_of_a_given_path(path: str, seq: str) -> float:
    if len(path) != len(seq):
        raise ValueError("Path and sequence must be of the same length")

    prob = 0.0
    for i in range(len(seq)):
        p1 = path[i]
        s1 = seq[i]
        if i == 0:
            prob += log(start_prob[p1])
        else:
            prob += log(trans_prob[path[i-1]][p1])
        prob += log(emit_prob[p1][s1])
    
    # Transition to End (only possible from I as in Nature Primer)
    last_state = path[-1]
    if last_state == 'I':
        prob += log(trans_prob['I']['end'])  # I → End

    return prob

# Define all the Parameters as in Nature Primer
states = ['E', '5', 'I']
start_prob = {'E': 1.0, '5': 0.0, 'I': 0.0}

trans_prob = {
    'E': {'E': 0.9, '5': 0.1},
    '5': {'I': 1.0},
    'I': {'I': 0.9, 'end': 0.1},
}

emit_prob = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}

path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

print("Log probability of given path:", round(get_log_prob_of_a_given_path(path, sequence),2))


Log probability of given path: -41.22


## Viterbi Algorithm

### 1. Initialization

- Set `V[i][s]` to store the highest log-probability of a path ending in state `s` at position `i`.
- Use `path[s]` to track the optimal path leading to state `s`.

### 2. Recursion (for i = 1 to n-1)

- For each position `i` and every possible current state (`curr_state`), check all previous states.
- Calculate the transition probability combined with the emission probability to find the maximum cumulative log-probability.
- Update both `V` (the score matrix) and `path` (the backpointer matrix).

### 3. Termination

- Identify the state with the highest score at the final position `i`.
- Retrieve the corresponding optimal path, and its log-probability will be the final score.

### 4. Output

- **best_path**: The most probable sequence of hidden states.
- **logp**: The log-probability of the best path.


In [8]:
V = [{}]
path = {}

# Initialization
for s in states:
    V[0][s] = log(start_prob[s]) + log(emit_prob[s].get(sequence[0], 0))
    path[s] = [s]

# Recursion
for i in range(1, len(sequence)):
    V.append({})
    newpath = {}

    for curr_state in states:
        max_prob = -math.inf
        best_prev_state = None

        for prev_state in states:
            # Get transition probability from prev_state to curr_state
            trans_p = trans_prob.get(prev_state, {}).get(curr_state, 0)

            # Only consider if a valid transition exists
            if trans_p > 0:
                # Get emission probability of current state emitting current observation
                emit_p = emit_prob[curr_state].get(sequence[i], 0)

                if emit_p > 0:
                    prob = V[i-1][prev_state] + math.log(trans_p) + math.log(emit_p)

                    if prob > max_prob:
                        max_prob = prob
                        best_prev_state = prev_state

        # Set the max probability for this current state at time i
        V[i][curr_state] = max_prob

        # Build the path
        if best_prev_state is not None:
            newpath[curr_state] = path[best_prev_state] + [curr_state]
        else:
            newpath[curr_state] = [curr_state]  # fallback in case of no valid path

    # Update path for the next iteration
    path = newpath

# Termination
n = len(sequence) - 1
(prob, state) = max((V[n][s], s) for s in states)

logp = round(prob, 2)
best_path = ''.join(path[state])


print("Viterbi best log probability:", logp)
print("Most likely path:", best_path)


Viterbi best log probability: -38.68
Most likely path: EEEEEEEEEEEEEEEEEEEEEEEEEE
